In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['SimHei']
rcParams['axes.unicode_minus'] = False
import seaborn as sns
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
#将数据输出结果导出
import os

In [ ]:
df=pd.read_csv('data/clean_2019-Nov.csv')
'''-------------同期群留存分析:按用户首次访问日期分组（同期群），看这群人在后续第 1、3、7、30 天的留存情况，衡量用户粘性、拉新质量，判断运营活动是否有长效价值。-------------'''

# 1. 统一日期格式
df['date'] = pd.to_datetime(df['date'])

# 2. 单独聚合每个用户的首次访问日期（生成实体表）
user_first = df.groupby('user_id')['date'].min().reset_index()
user_first.columns = ['user_id', 'first_date']

# 3. 左连接回主表（替代transform，生成普通物理列）
df = df.merge(user_first, on='user_id', how='left')

# 4. 计算距离首次访问的天数
df['days_since_first'] = (df['date'] - df['first_date']).dt.days

# 5. 构建同期群数据
cohort = df.groupby(['first_date', 'days_since_first'])['user_id'].nunique().reset_index()

# 6. 生成透视表（列名转字符串，规避数值列名的索引问题）
cohort_pivot = cohort.pivot(
    index='first_date',
    columns='days_since_first',
    values='user_id'
)
cohort_pivot.columns = cohort_pivot.columns.astype(str)
#  计算留存率（除以第0天的用户数）
cohort_pivot.index = cohort_pivot.index.astype(str).str[:10]
cohort_retention = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0)

#  画留存热力图
plt.figure(figsize=(12, 8))
sns.heatmap(cohort_retention, annot=True, fmt='.1%', cmap='YlGnBu')
plt.title('用户同期群留存热力图')
plt.show()
print(" 同期群留存分析结果")
print("="*50)

# 输出留存率表前5行概览
print("留存率矩阵（前5个批次）：")
print(cohort_retention.head().round(4).to_string())
print("-"*30)

# 计算关键节点的平均留存率
day_1_ret = cohort_retention.iloc[:, 1].mean() if cohort_retention.shape[1] > 1 else 0
day_3_ret = cohort_retention.iloc[:, 3].mean() if cohort_retention.shape[1] > 3 else 0
day_7_ret = cohort_retention.iloc[:, 7].mean() if cohort_retention.shape[1] > 7 else 0
day_30_ret = cohort_retention.iloc[:, 30].mean() if cohort_retention.shape[1] > 30 else 0

print(f"平均次日留存（第1天）：{day_1_ret:.4f}（{day_1_ret*100:.2f}%）")
print(f"平均3日留存：{day_3_ret:.4f}（{day_3_ret*100:.2f}%）")
print(f"平均7日留存：{day_7_ret:.4f}（{day_7_ret*100:.2f}%）")
print(f"平均30日留存：{day_30_ret:.4f}（{day_30_ret*100:.2f}%）")
# ====================== 保存数据用于后续可视化输出 ======================
# 留存率矩阵（重置索引方便读取）
cohort_retention.reset_index().to_csv('output/data/user_retention.csv',
                                      index=False, encoding='utf-8-sig')